# Make_Catalog

The entry point for AI agents and other machines, at the repository root:

- `catalog.ttl`: DCAT + VoID. Every published graph and file with its licence, version, triple count and sha256; one VoID linkset per link of `schema/links.yaml`.
- `llms.txt`: the same in the llms.txt layout, with the rules an agent should keep.

The catalogue lists `schema/vocabulary.ttl` (written by `checks/00_Schema`); section 4 checks it against the built graphs.

Both come only from what the repository already states: `Reference_Versions.json`, the Provenance files of `base/` and `linked/`, and `schema/links.yaml`. Run it after Make_Links and Make_COSMoS_Model.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from rdflib import Graph
from rdflib.namespace import DCAT

from cfa import SCHEMA_DIR
from cfa.catalog import PREFIXES, SITE, catalog, llms_txt
from cfa.link import load_links

## 1. Build

In [ ]:
specs = load_links(SCHEMA_DIR / 'links.yaml')
g = catalog(REPO_ROOT, specs)
text = llms_txt(REPO_ROOT, specs, g)
print(len(g), 'triples in the catalogue;', len(specs), 'linksets')

## 2. Write

In [ ]:
for k, v in PREFIXES.items():
    g.bind(k, v)
(REPO_ROOT / 'catalog.ttl').write_text(g.serialize(format='longturtle'), encoding='utf-8')
(REPO_ROOT / 'llms.txt').write_text(text, encoding='utf-8')

## 3. Check

Read `catalog.ttl` back: it must be the same graph. Every download URL must name a file in the repository.

In [ ]:
back = Graph().parse(REPO_ROOT / 'catalog.ttl', format='turtle')
if len(back) != len(g):
    raise ValueError(f'catalog.ttl read back has {len(back)} triples, built {len(g)}')
urls = [str(u) for u in back.objects(None, DCAT.downloadURL)]
missing = [u for u in urls if not (REPO_ROOT / u[len(SITE):].replace('%20', ' ')).exists()]
if missing:
    raise ValueError(f'download URLs without a file: {missing}')
print(len(urls), 'download URLs, every one a file in the repository')

## 4. Check the vocabulary against the graphs

Every class and predicate of the base graphs and the link graphs (the files linked/Provenance.json lists, the local sign-in ones included) must be declared in `schema/vocabulary.ttl`. A declared column that no triple uses is listed: a column that is empty in every row gives no triple.

In [ ]:
import json

from cfa.vocabulary import unused_and_undeclared

prov = json.loads((REPO_ROOT / 'linked' / 'Provenance.json').read_text(encoding='utf-8'))
files = list(prov['inputs']) + ['linked/' + info['file'] for info in prov['link_graphs'].values()]
vocab = Graph().parse(REPO_ROOT / 'schema' / 'vocabulary.ttl')
unused, undeclared = unused_and_undeclared(vocab, [Graph().parse(REPO_ROOT / f) for f in files])
if undeclared:
    raise ValueError(f'used in the graphs, not in the vocabulary: {undeclared}')
print(len(files), 'graphs: every class and predicate is declared')
print('declared, used by no triple:', [str(u) for u in unused])